# AG_PRAXIS NB04 — Leakage Diagnostic

Every class in this dataset was recorded in its own session, and several classes were
recorded in more than one. A row therefore carries two things a model could learn: what
the traffic was doing, and which recording it came from. Those two are almost the same
thing here, because a recording holds one class, so a model that reads the recording
gets the class for free and never has to learn the attack.

The capture inventory this repository already holds rules out the crude version of that
problem. No column holds a single value throughout a capture while differing between
captures, so nothing in the file is acting as a name for the session. What it does not
rule out is the distributional version. `Header_Length` averages around 59.6 in one
DDoS-ICMP capture and around 226.0 in another, and both of those recordings are the same
attack. A mean that moves by a factor of four between two recordings of one thing is
enough to tell the two recordings apart, even though no single row announces where it
came from.

So the question this notebook asks is whether the features identify the recording. It
answers it four ways. First it ranks every feature by how far it shifts between captures
relative to how much it varies inside one. Then it trains a classifier whose target is
the capture itself, on all features, on each feature family separately, and on the five
features that published work on this dataset reports as most important. Last it
classifies the attack twice with the same model, once holding out whole captures and
once pooling the captures and splitting rows at random, because the gap between those
two numbers is what the leakage is worth in macro-F1.

Everything written here goes to the artefacts folder on Drive. A notebook running in
Colab cannot commit, so a file written into the cloned repository lives exactly as long
as the session does. The two files that carry the result are printed back at the end so
they stay attached to the run that produced them.

Mount Drive, clone the repository so that `src/` can be imported, and record the commit
the results belong to. The repository is read from and never written to.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Paths, the seed and the model settings come from `config/base.yaml`. The column list
comes from `data/processed/schema.json` and the class to capture mapping and tiers come
from `data/processed/capture_inventory.json`, so this notebook uses the inventory that
was checked rather than re-deriving one and quietly disagreeing with it. If either file
is missing there is nothing to stand on, so it stops.

`FAST` changes two things and it is worth being exact about which. It changes how much
of each file is read, which changes the variance ranking, and it changes how many rows
are available to sample from for the models. At `FAST=1` the reader takes the opening
one percent of each file, and the opening rows of a session are a contiguous slice of it
rather than a sample of it, so a session can look more uniform than it is. Every number
below is a screen until this runs at `FAST=0`.

Two thresholds are fixed here, before any result exists, so that the verdict at the end
is a rule applied rather than a number interpreted after the fact. Capture identity
counts as recoverable if the capture classifier scores above 0.5 accuracy and at least
five times the chance rate.

In [ ]:
import json
import random
import time

import numpy as np
import pandas as pd
import yaml
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from src import captures as cap
from src import inventory as inv
from src import leakage as leak
from src import runs as runlog

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
TRAIN_DIR = Path(CFG["paths"]["train_dir"])
TEST_DIR = Path(CFG["paths"]["test_dir"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])
OUT_DIR = ARTIFACTS / "NB04"

PROCESSED = REPO_ROOT / "data" / "processed"
for required in ("schema.json", "capture_inventory.json"):
    if not (PROCESSED / required).exists():
        raise FileNotFoundError(f"{PROCESSED / required} not found in the repository.")

SCHEMA = json.loads((PROCESSED / "schema.json").read_text())
INVENTORY = json.loads((PROCESSED / "capture_inventory.json").read_text())

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )
OUT_DIR.mkdir(parents=True, exist_ok=True)

FAST = os.environ.get("FAST", "1") == "1"
READ_FRAC = CFG["fast_mode"]["subsample_frac"] if FAST else 1.0

# The models get an equal draw from every capture, so that a capture with more rows
# cannot be identified by being common. The cap also keeps the fits affordable. The
# variance ranking does not use this draw; it uses every row that was read.
ROWS_PER_CAPTURE = 1_000 if FAST else 5_000

N_TREES = 100
TEST_FRACTION = 0.25

# Fixed before any result exists.
RECOVERABLE_ACCURACY = 0.50
RECOVERABLE_MULTIPLE_OF_CHANCE = 5.0

TIER_A = INVENTORY["tiers"]["A"]
TIER_B = INVENTORY["tiers"]["B"]
TIER_OF = {label: entry["tier"] for label, entry in INVENTORY["classes"].items()}
ROW_COUNTS = INVENTORY["rows_per_capture_file"]

pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

print(f"seed       : {SEED}")
print(f"train dir  : {TRAIN_DIR}   exists={TRAIN_DIR.exists()}")
print(f"test dir   : {TEST_DIR}   exists={TEST_DIR.exists()}")
print(f"output dir : {OUT_DIR}")
print()
print(f"schema.json            written on {SCHEMA['generated_on']} at sha {SCHEMA['git_sha']}")
print(f"capture_inventory.json written on {INVENTORY['generated_on']} at sha {INVENTORY['git_sha']}")
print(f"  classes      : {INVENTORY['n_classes']} in {INVENTORY['n_files']} files")
print(f"  tier A       : {len(TIER_A)} classes, more than one recording each")
print(f"  tier B       : {len(TIER_B)} classes, one recording each")
if INVENTORY.get("fast_mode"):
    print("  That run had FAST=1. Its row counts are exact; its column checks were a screen.")
print()
print(f"FAST                 : {int(FAST)}")
print(f"reads                : {'every row of every file' if not FAST else f'the first {READ_FRAC:.0%} of each file'}")
print(f"rows per capture kept for models : {ROWS_PER_CAPTURE:,}")
print(f"forest               : {N_TREES} trees, seed {SEED}")
print(f"recoverable if       : accuracy > {RECOVERABLE_ACCURACY} and > {RECOVERABLE_MULTIPLE_OF_CHANCE}x chance")
if FAST:
    print()
    print("FAST=1. The opening rows of a file are a contiguous slice of a session, not a")
    print("sample of it, so treat every number below as a screen. Steps 3, 4 and 7 need")
    print("FAST=0 before the result counts.")

Models are built here, so the seed is set before anything else runs. The generator drawn
from it is the one that chooses which rows each capture contributes, so the same rows are
read on a re-run.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)
print(f"seeded with {SEED}")

Two columns come out before anything else touches the data. The inventory records
`DHCP` and `Drate` as holding one value across the entire corpus, and a column with no
variation carries no information about anything. Leaving them in would not change a
result, but it would put two features in every importance table that cannot possibly
matter, and it would make the family counts wrong.

In [ ]:
DROPPED = list(INVENTORY["columns_constant_everywhere"])
EXPECTED_DROPPED = ["DHCP", "Drate"]
FEATURES = [c for c in SCHEMA["columns"] if c not in DROPPED]

print(f"columns in the files       : {len(SCHEMA['columns'])}")
print(f"constant across the corpus : {DROPPED}")
print(f"features used here         : {len(FEATURES)}")

if sorted(DROPPED) != sorted(EXPECTED_DROPPED):
    print()
    print(f"The inventory names {DROPPED}, not {EXPECTED_DROPPED}. Read it before continuing.")

missing = [c for c in EXPECTED_DROPPED if c in FEATURES]
assert not missing, f"{missing} should have been dropped and was not"
print()
print(", ".join(FEATURES))

Step 5 asks which family of features carries the signal, so the features have to be
sorted into families first. The rule is in `src/inventory.py` and works from column names
alone, which is a guess about what a column measures rather than knowledge of it. If
`config/feature_families.yaml` has been filled in and marked reviewed, that file wins.

One assignment is worth flagging before it produces a misleading answer. `Header_Length`
matches the protocol rule on the word header and the statistical rule on the word length,
and the first match wins, so it sits under protocol. It is also the feature the opening
paragraph names as the leading suspect. If protocol scores highly in step 5, the next
question is whether it is the family or that one column, which is what step 6 is for.

In [ ]:
FAMILY_FILE = REPO_ROOT / "config" / "feature_families.yaml"
raw_families = yaml.safe_load(FAMILY_FILE.read_text()) if FAMILY_FILE.exists() else None

if raw_families and raw_families.get("families") and raw_families.get("status") == "reviewed":
    FAMILIES = {
        name: [c for c in columns if c in FEATURES]
        for name, columns in raw_families["families"].items()
    }
    AMBIGUOUS = raw_families.get("ambiguous", {})
    FAMILY_SOURCE = "config/feature_families.yaml, reviewed"
else:
    assignment = inv.assign_families(FEATURES)
    FAMILIES = {name: columns for name, columns in assignment["families"].items() if columns}
    AMBIGUOUS = assignment["ambiguous"]
    FAMILY_SOURCE = "src.inventory.assign_families, from column names"

FAMILY_OF = {c: name for name, columns in FAMILIES.items() for c in columns}
TEST_FAMILIES = [name for name in ("timing", "protocol", "statistical") if FAMILIES.get(name)]

print(f"families from : {FAMILY_SOURCE}")
print()
for name, columns in FAMILIES.items():
    print(f"{name:<12} {len(columns):>2}   {', '.join(columns)}")
print()
unassigned = [c for c in FEATURES if c not in FAMILY_OF]
print(f"features in no family : {unassigned or 'none'}")
print(f"families tested separately in step 5 : {', '.join(TEST_FAMILIES)}")
print()
if AMBIGUOUS:
    print("matched more than one rule and went to the first match:")
    for column, matched in AMBIGUOUS.items():
        print(f"  {column:<16} {matched}  ->  {FAMILY_OF.get(column)}")

Now the files, and the one decision that everything after it depends on: what counts as
one recording.

For a class recorded once, the two shipped files are the two halves of a single session,
so they are one recording and get one key. For a class recorded several times, the chunk
numbers restart in each partition, so a train chunk and a test chunk carrying the same
number are two different sessions and the partition has to stay in the key.

That claim is checkable from the row counts alone, which is what the next cell prints. A
shipped test file that is the held-out tail of a recording is a fraction of its training
file. A shipped test file that is its own recording is about the same size as one chunk
of that class. The two patterns are far enough apart to tell without reading a row.

In [ ]:
train_files = sorted(TRAIN_DIR.glob("*.csv"))
test_files = sorted(TEST_DIR.glob("*.csv"))
ALL_FILES = train_files + test_files

files = pd.DataFrame([{"file": p.name, "path": str(p), **cap.parse_capture(p.name)} for p in ALL_FILES])
files["tier"] = files["label"].map(TIER_OF)
files["rows"] = files["file"].map(ROW_COUNTS)
files["capture"] = [
    leak.capture_key(r.capture_id, r.partition, r.tier) for r in files.itertuples(index=False)
]

unknown_tier = files.loc[files["tier"].isna(), "label"].unique().tolist()
assert not unknown_tier, f"labels missing from the inventory: {unknown_tier}"

size_check = []
for label, group in files.groupby("label", sort=True):
    train_rows = group.loc[group["partition"] == "train", "rows"]
    test_rows = group.loc[group["partition"] == "test", "rows"]
    if train_rows.empty or test_rows.empty:
        continue
    size_check.append(
        {
            "label": label,
            "tier": TIER_OF[label],
            "train_files": len(train_rows),
            "mean_train_file": int(train_rows.mean()),
            "mean_test_file": int(test_rows.mean()),
            "test_over_train": round(float(test_rows.mean() / train_rows.mean()), 3),
        }
    )

size_check = pd.DataFrame(size_check).sort_values(["tier", "label"])
print("size of a shipped test file against the size of one training file of the same class")
print(size_check.to_string(index=False))
tier_a_ratio = size_check.loc[size_check["tier"] == "A", "test_over_train"]
tier_b_ratio = size_check.loc[size_check["tier"] == "B", "test_over_train"]

print()
print(f"tier A : {tier_a_ratio.min():.2f} to {tier_a_ratio.max():.2f}, median {tier_a_ratio.median():.2f}")
print(f"tier B : {tier_b_ratio.min():.2f} to {tier_b_ratio.max():.2f}, median {tier_b_ratio.median():.2f}")
print()
if tier_a_ratio.min() > tier_b_ratio.max():
    print("The two ranges do not overlap. A tier B test file is a fraction of its training")
    print("file, which is what a cut inside one recording looks like. A tier A test file is")
    print("close to the size of a whole training chunk, which a percentage split cannot")
    print("produce, so it is its own recording and is keyed as one.")
else:
    print("The two ranges overlap, so file size does not settle which tier A test files are")
    print("separate recordings. The keys below assume they are, and that assumption is")
    print("carrying more weight than this evidence supports.")
print()
print(f"files    : {len(files)}")
print(f"captures : {files['capture'].nunique()}   ({files.loc[files['tier'] == 'A', 'capture'].nunique()} in tier A)")
print()
print(files.loc[files["tier"] == "A", ["file", "capture", "label", "partition", "rows"]].to_string(index=False))

One pass over the files does both jobs. While a file is open, the mean and variance of
every feature in it are computed over every row that was read, and those three numbers
per feature are all the variance ranking needs, so the rows never have to be held in
memory together. Then a fixed number of rows is drawn at random from the file and kept
for the models.

The draw is the same size for every capture. A capture with more rows would otherwise be
easier to guess simply by being common, and an accuracy inflated that way would say
nothing about the features. It also keeps the forests affordable, which matters because
there are a dozen of them below.

Rows holding a non-finite value are dropped, and the count is printed rather than
silently absorbed, because a feature with infinities in it can dominate a variance
ranking by itself.

In [ ]:
moment_rows = []
sample_frames = []
rows_read = 0
rows_dropped = 0

started = time.perf_counter()
for i, row in enumerate(files.itertuples(index=False), start=1):
    frame = cap.read_capture(row.path, frac=READ_FRAC, known_rows=ROW_COUNTS.get(row.file))
    frame = frame[FEATURES].astype("float32")
    rows_read += len(frame)

    keep = leak.finite_rows(frame)
    rows_dropped += int((~keep).sum())
    frame = frame.loc[keep]

    m = leak.moments(frame, FEATURES)
    m["file"] = row.file
    m["capture"] = row.capture
    m["label"] = row.label
    m["tier"] = row.tier
    moment_rows.append(m)

    kept = leak.sample_rows(frame, ROWS_PER_CAPTURE, RNG).copy()
    kept["__file"] = row.file
    kept["__capture"] = row.capture
    kept["__label"] = row.label
    kept["__tier"] = row.tier
    sample_frames.append(kept)

    del frame
    if i % 10 == 0 or i == len(files):
        print(f"  {i:>3}/{len(files)} files   {rows_read:,} rows read   {time.perf_counter() - started:,.0f}s")

MOMENTS = pd.concat(moment_rows, ignore_index=True)
SAMPLE = pd.concat(sample_frames, ignore_index=True)

print()
print(f"rows read            : {rows_read:,} of {sum(ROW_COUNTS.values()):,} in the corpus")
print(f"rows with a non-finite value, dropped : {rows_dropped:,}")
print(f"rows kept for models : {len(SAMPLE):,}   ({SAMPLE[FEATURES].memory_usage().sum() / 1e6:,.0f} MB of features)")
print(f"captures represented : {SAMPLE['__capture'].nunique()}")
print()
print("rows kept per capture, tier A:")
print(SAMPLE.loc[SAMPLE["__tier"] == "A", "__capture"].value_counts().sort_index().to_string())

## Step 3 — how far each feature moves between recordings

For every feature I have a mean and a variance inside each capture. The spread of those
per-capture means is the between-capture variance, and the average of the per-capture
variances is the within-capture variance. Their ratio is the number this step ranks on.
A feature with a large ratio sits in a different place in each recording while holding
fairly still inside each one, which is precisely the shape a model would read to tell
recordings apart.

The ranking over all captures has one weakness that has to be named, because otherwise
it reads as stronger evidence than it is. A capture holds exactly one class, so a
feature that separates the classes well also separates the captures well, and it will
rank high for a reason that has nothing to do with leakage. The second column fixes
that: the same ratio computed inside one class at a time, using only the classes
recorded more than once, so the class is held fixed and what is left is the difference
between recordings of the same attack. That column is the one that speaks to
provenance. Both are kept, because the difference between them is informative on its
own.

In [ ]:
POOLED = leak.pool_moments(MOMENTS, by="capture")
capture_labels = files.drop_duplicates("capture").set_index("capture")["label"]
capture_tiers = files.drop_duplicates("capture").set_index("capture")["tier"]

ranking = leak.variance_ratio(POOLED)
by_class = leak.variance_ratio_by_class(POOLED, capture_labels, min_captures=2)

RANKING = (
    ranking.join(by_class[["n_classes", "within_class_median_ratio", "within_class_max_ratio"]])
    .reset_index()
    .rename(columns={"index": "feature"})
)
RANKING["family"] = RANKING["feature"].map(FAMILY_OF)
positive_min = RANKING["min_capture_mean"].where(RANKING["min_capture_mean"] > 0)
RANKING["fold_range_of_means"] = RANKING["max_capture_mean"] / positive_min
RANKING.insert(0, "rank", np.arange(1, len(RANKING) + 1))

SHOW = [
    "rank",
    "feature",
    "family",
    "ratio",
    "within_class_median_ratio",
    "between_var",
    "within_var",
    "min_capture_mean",
    "max_capture_mean",
]
printable = RANKING[SHOW].copy()
for column in printable.columns:
    if printable[column].dtype.kind == "f":
        printable[column] = printable[column].map(lambda v: f"{v:,.4g}")

print(f"between-capture variance over mean within-capture variance, {POOLED['capture'].nunique()} captures")
print(f"the within-class column pools {int(by_class['n_classes'].max()) if len(by_class) else 0} classes recorded more than once")
print()
print(printable.to_string(index=False))

The head of that table is the answer to step 3, and the two columns should be read
together. A feature high on both is one that moves between recordings of the same
attack, which is what a capture-identifying model would use. A feature high on the first
and low on the second is separating classes, which is what a feature is supposed to do.

In [ ]:
top = RANKING.head(10)
print("ten features that shift most between captures")
print()
for _, row in top.iterrows():
    fold = row["fold_range_of_means"]
    fold_text = f"{fold:,.1f}x" if np.isfinite(fold) else "not a ratio, the smallest capture mean is not positive"
    print(f"{row['rank']:>2}. {row['feature']:<16} {row['family']:<12} ratio {row['ratio']:>10,.2f}   within class {row['within_class_median_ratio']:>10,.2f}")
    print(f"     capture means run {row['min_capture_mean']:,.4g} to {row['max_capture_mean']:,.4g}, a spread of {fold_text}")
print()

within_class_top = RANKING.sort_values("within_class_median_ratio", ascending=False).head(10)
print("ten features that shift most between recordings of the same attack")
print()
print(
    within_class_top[["feature", "family", "within_class_median_ratio", "ratio"]]
    .to_string(index=False, float_format=lambda v: f"{v:,.2f}")
)

In [ ]:
variance_doc = {
    "generated_by": "AG_PRAXIS_NB04_leakage_diagnostic.ipynb",
    "generated_on": RUN_DATE,
    "git_sha": GIT_SHA,
    "fast_mode": FAST,
    "read_fraction": READ_FRAC,
    "rows_read": int(rows_read),
    "rows_in_corpus": int(sum(ROW_COUNTS.values())),
    "rows_dropped_non_finite": int(rows_dropped),
    "features_dropped": DROPPED,
    "n_features": len(FEATURES),
    "n_captures": int(POOLED["capture"].nunique()),
    "definition": {
        "between_var": "variance of the per-capture means, one mean per capture",
        "within_var": "mean of the per-capture variances",
        "ratio": "between_var / within_var, infinite if a feature is constant inside every capture",
        "within_class_median_ratio": "the same ratio computed inside each class recorded more than once, median across those classes",
    },
    "ranking": [
        {
            "rank": int(row["rank"]),
            "feature": row["feature"],
            "family": row["family"],
            "ratio": leak.json_number(row["ratio"]),
            "within_class_median_ratio": leak.json_number(row["within_class_median_ratio"]),
            "within_class_max_ratio": leak.json_number(row["within_class_max_ratio"]),
            "n_classes_in_within_class_ratio": int(row["n_classes"]) if pd.notna(row["n_classes"]) else 0,
            "between_var": leak.json_number(row["between_var"]),
            "within_var": leak.json_number(row["within_var"]),
            "grand_mean": leak.json_number(row["grand_mean"]),
            "min_capture_mean": leak.json_number(row["min_capture_mean"]),
            "max_capture_mean": leak.json_number(row["max_capture_mean"]),
        }
        for _, row in RANKING.iterrows()
    ],
}

VARIANCE_PATH = OUT_DIR / "variance_ranking.json"
VARIANCE_PATH.write_text(json.dumps(cap.jsonable(variance_doc), indent=2, default=str) + "\n")
print(f"wrote {VARIANCE_PATH}   {VARIANCE_PATH.stat().st_size:,} bytes")

## Step 4 — can a model name the recording

The ranking says which features shift between recordings. It does not say whether the
shift is large enough to identify one, because a model reads all of them together and
small differences combine. So this asks the question directly: give a forest every
feature and ask it not what the attack was but which capture the row came from.

Only the eight tier A classes are used, because they are the classes recorded more than
once. The target is the capture key, so the classes with several recordings contribute
several targets, and a model that answers correctly has separated recordings of the same
attack from each other rather than just separated the attacks.

There is nothing to exclude from the input. The class of a row is carried by the name of
the file it came from and by no column inside it, so there is no label column to leak
through. The rows are split at random within each capture, which means every capture
appears in both halves. That is the correct split here: the question is whether the
features encode capture identity at all, not whether that encoding generalises to a
recording the model has never seen.

The accuracy has to be read against the two baselines the metrics carry. Chance is one
over the number of captures, and the majority rate is what always naming the largest
capture would give.

In [ ]:
RESULTS = {}

A = SAMPLE.loc[SAMPLE["__tier"] == "A"].reset_index(drop=True)
X_A = A[FEATURES].to_numpy(dtype="float32")
y_capture = A["__capture"].to_numpy()
y_label = A["__label"].to_numpy()

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_A, y_capture, test_size=TEST_FRACTION, random_state=SEED, stratify=y_capture
)

print(f"tier A rows        : {len(A):,}")
print(f"captures to name   : {len(np.unique(y_capture))}")
print(f"train / test rows  : {len(yc_train):,} / {len(yc_test):,}")
print("fitting")

RESULTS["capture_id_all"] = runlog.fit_and_save(
    OUT_DIR,
    "capture_id_all_features",
    RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1),
    Xc_train,
    yc_train,
    Xc_test,
    yc_test,
    features=FEATURES,
    target="capture_id",
    notes="tier A only, all features, rows split at random within each capture",
    extra_config={"fast_mode": FAST, "rows_per_capture": ROWS_PER_CAPTURE, "seed": SEED, "git_sha": GIT_SHA},
)

In [ ]:
record = RESULTS["capture_id_all"]
m = record["metrics"]

print(f"target            : which capture the row came from, {m['n_classes']} captures")
print(f"accuracy          : {m['accuracy']:.4f}")
print(f"macro-F1          : {m['macro_f1']:.4f}")
print(f"weighted-F1       : {m['weighted_f1']:.4f}")
print(f"chance            : {m['chance_rate']:.4f}")
print(f"largest capture   : {m['majority_class_rate']:.4f}")
print(f"accuracy / chance : {m['accuracy'] / m['chance_rate']:,.1f}x")
print(f"train seconds     : {m['train_seconds']:,.1f}")
print(f"saved to          : {record['run_dir']}")
print()
print("top 15 features by importance")
for i, entry in enumerate(m["top_features"], start=1):
    family = FAMILY_OF.get(entry["feature"], "")
    print(f"  {i:>2}. {entry['feature']:<16} {family:<12} {entry['importance']:.4f}")
print()
share = sum(e["importance"] for e in m["top_features"])
print(f"those 15 hold {share:.1%} of the total importance")

That number is about capture identity and class identity together, and the two are worth
separating. A capture belongs to one class, so a model can get part of the way by
recognising the attack and then guessing among that class's recordings. The stricter
question is whether it can tell one recording of DDoS-ICMP from another recording of
DDoS-ICMP, with the class already known.

So the same fit is repeated inside each tier A class separately. Chance is now one over
the number of recordings of that class, four or ten rather than fifty, so the accuracy
is not comparable with the number above and is read against its own baseline. Each class
is saved as it finishes, which is what keeps a dropped session from costing the whole
loop.

In [ ]:
within_class_runs = {}
for label in sorted(A["__label"].unique()):
    rows = A.loc[A["__label"] == label]
    captures = rows["__capture"].nunique()
    if captures < 2:
        continue
    X_one = rows[FEATURES].to_numpy(dtype="float32")
    y_one = rows["__capture"].to_numpy()
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_one, y_one, test_size=TEST_FRACTION, random_state=SEED, stratify=y_one
    )
    print(f"fitting {label}: {captures} recordings, {len(y_tr):,} train rows")
    within_class_runs[label] = runlog.fit_and_save(
        OUT_DIR,
        f"capture_id_within_class_{label}",
        RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1),
        X_tr,
        y_tr,
        X_te,
        y_te,
        features=FEATURES,
        target="capture_id",
        notes=f"captures of {label} only, class held fixed",
        extra_config={"label": label, "fast_mode": FAST, "seed": SEED, "git_sha": GIT_SHA},
    )

In [ ]:
RESULTS["capture_id_within_class"] = within_class_runs

rows = []
for label, run in within_class_runs.items():
    m = run["metrics"]
    rows.append(
        {
            "label": label,
            "recordings": m["n_classes"],
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
            "chance": m["chance_rate"],
            "over_chance": m["accuracy"] / m["chance_rate"],
            "top_feature": m["top_features"][0]["feature"],
        }
    )

within_class_table = pd.DataFrame(rows)
WITHIN_CLASS_MEAN_ACC = float(within_class_table["accuracy"].mean())

print("naming the recording with the attack class already known")
print()
print(within_class_table.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
print()
print(f"mean accuracy across the eight classes : {WITHIN_CLASS_MEAN_ACC:.4f}")
print(f"mean chance rate                       : {within_class_table['chance'].mean():.4f}")

## Step 5 — which family carries it

The same test, restricted to one family of features at a time. A family that scores near
the full-feature result carries the provenance signal on its own; a family that scores
near chance does not carry it at all. Everything else about the run is held fixed, so
the accuracy differences are attributable to the feature set and nothing else.

Each family is saved as its fit finishes rather than after the loop.

In [ ]:
family_runs = {}
for family in TEST_FAMILIES:
    columns = FAMILIES[family]
    X_family = A[columns].to_numpy(dtype="float32")
    Xf_train, Xf_test, yf_train, yf_test = train_test_split(
        X_family, y_capture, test_size=TEST_FRACTION, random_state=SEED, stratify=y_capture
    )
    print(f"fitting {family}: {len(columns)} features")
    family_runs[family] = runlog.fit_and_save(
        OUT_DIR,
        f"capture_id_family_{family}",
        RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1),
        Xf_train,
        yf_train,
        Xf_test,
        yf_test,
        features=columns,
        target="capture_id",
        notes=f"tier A only, {family} features only",
        extra_config={"family": family, "fast_mode": FAST, "seed": SEED, "git_sha": GIT_SHA},
    )

In [ ]:
RESULTS["capture_id_family"] = family_runs

all_features_accuracy = RESULTS["capture_id_all"]["metrics"]["accuracy"]
rows = [
    {
        "features": "all",
        "n_features": len(FEATURES),
        "accuracy": all_features_accuracy,
        "macro_f1": RESULTS["capture_id_all"]["metrics"]["macro_f1"],
        "share_of_all": 1.0,
        "top_feature": RESULTS["capture_id_all"]["metrics"]["top_features"][0]["feature"],
    }
]
for family, run in family_runs.items():
    m = run["metrics"]
    rows.append(
        {
            "features": family,
            "n_features": run["config"]["n_features"],
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
            "share_of_all": m["accuracy"] / all_features_accuracy,
            "top_feature": m["top_features"][0]["feature"],
        }
    )

family_table = pd.DataFrame(rows)
CARRYING_FAMILY = max(family_runs, key=lambda f: family_runs[f]["metrics"]["accuracy"])

print(f"naming the capture, {len(np.unique(y_capture))} recordings, chance {1 / len(np.unique(y_capture)):.4f}")
print()
print(family_table.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
print()
print(f"strongest family : {CARRYING_FAMILY}, {family_runs[CARRYING_FAMILY]['metrics']['accuracy']:.4f} accuracy from {family_runs[CARRYING_FAMILY]['config']['n_features']} features")

## Step 6 — the five features prior work names

Published work on this dataset reports `IAT`, `Rate`, `Srate`, `Header_Length` and
`rst_count` as the five features that matter most for classifying the attack. If those
same five can name the recording, then the features driving the published result are the
features carrying provenance, and the two questions are not separable in that model.

Two fits. The five named features, then the three of them that are timing measurements,
because timing is the family most likely to reflect the conditions of a session rather
than the behaviour of an attack.

In [ ]:
NAMED_SUSPECTS = ["IAT", "Rate", "Srate", "Header_Length", "rst_count"]
TIMING_THREE = ["IAT", "Rate", "Srate"]

missing_suspects = [c for c in NAMED_SUSPECTS if c not in FEATURES]
assert not missing_suspects, f"named features not present: {missing_suspects}"

print("named features and where each sits in the step 3 ranking")
lookup = RANKING.set_index("feature")
for column in NAMED_SUSPECTS:
    row = lookup.loc[column]
    print(f"  {column:<16} {FAMILY_OF.get(column):<12} rank {int(row['rank']):>2} of {len(RANKING)}   ratio {row['ratio']:,.2f}   within class {row['within_class_median_ratio']:,.2f}")
print()

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    A[NAMED_SUSPECTS].to_numpy(dtype="float32"),
    y_capture,
    test_size=TEST_FRACTION,
    random_state=SEED,
    stratify=y_capture,
)
print("fitting the five named features")

RESULTS["capture_id_named_five"] = runlog.fit_and_save(
    OUT_DIR,
    "capture_id_named_five",
    RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1),
    Xs_train,
    ys_train,
    Xs_test,
    ys_test,
    features=NAMED_SUSPECTS,
    target="capture_id",
    notes="the five features published work reports as most important",
    extra_config={"fast_mode": FAST, "seed": SEED, "git_sha": GIT_SHA},
)

In [ ]:
Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    A[TIMING_THREE].to_numpy(dtype="float32"),
    y_capture,
    test_size=TEST_FRACTION,
    random_state=SEED,
    stratify=y_capture,
)
print("fitting the three timing features among them")

RESULTS["capture_id_timing_three"] = runlog.fit_and_save(
    OUT_DIR,
    "capture_id_timing_three",
    RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1),
    Xt_train,
    yt_train,
    Xt_test,
    yt_test,
    features=TIMING_THREE,
    target="capture_id",
    notes="IAT, Rate and Srate only",
    extra_config={"fast_mode": FAST, "seed": SEED, "git_sha": GIT_SHA},
)

In [ ]:
named = RESULTS["capture_id_named_five"]["metrics"]
timing = RESULTS["capture_id_timing_three"]["metrics"]

comparison = pd.DataFrame(
    [
        {"features": "all", "n": len(FEATURES), "accuracy": all_features_accuracy, "macro_f1": RESULTS["capture_id_all"]["metrics"]["macro_f1"]},
        {"features": "the five named", "n": len(NAMED_SUSPECTS), "accuracy": named["accuracy"], "macro_f1": named["macro_f1"]},
        {"features": "IAT, Rate, Srate", "n": len(TIMING_THREE), "accuracy": timing["accuracy"], "macro_f1": timing["macro_f1"]},
    ]
)
comparison["share_of_all"] = comparison["accuracy"] / all_features_accuracy

print(comparison.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
print()
print("importance within the five named features")
for entry in named["top_features"]:
    print(f"  {entry['feature']:<16} {entry['importance']:.4f}")

## Step 7 — what it costs in macro-F1

The tests so far ask what the features encode. This one asks what it is worth, by
classifying the attack twice with the same forest and changing only how the rows are
divided.

The first split holds out a whole recording of each class, the last one by name, and
trains on the others. Nothing the model saw was recorded in the session it is tested on.
The second pools the same rows and splits them at random, so every recording appears in
both halves and the model can have seen rows from the session it is being tested on. The
test fraction is set to match the first split, so the two differ in how the rows were
divided and in nothing else.

Tier A only, eight classes. That restriction is the whole point: these are the classes
where a clean comparison is possible, because a class recorded once cannot have a
recording held out.

In [ ]:
HOLDOUT = leak.holdout_captures(A, capture_col="__capture", label_col="__label")

is_holdout = A["__capture"].isin(HOLDOUT.values()).to_numpy()
HOLDOUT_FRACTION = float(is_holdout.mean())

print("held out, one recording per class")
for label in sorted(HOLDOUT):
    trained_on = sorted(A.loc[(A["__label"] == label) & ~A["__capture"].isin([HOLDOUT[label]]), "__capture"].unique())
    print(f"  {label:<12} test on {HOLDOUT[label]:<28} train on {len(trained_on)} others")
print()
print(f"held-out rows : {int(is_holdout.sum()):,} of {len(A):,}, {HOLDOUT_FRACTION:.1%}")
print("fitting")

RESULTS["attack_capture_holdout"] = runlog.fit_and_save(
    OUT_DIR,
    "attack_tier_a_capture_holdout",
    RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1),
    X_A[~is_holdout],
    y_label[~is_holdout],
    X_A[is_holdout],
    y_label[is_holdout],
    features=FEATURES,
    target="label",
    notes="tier A only, the last recording of each class held out entirely",
    extra_config={
        "protocol": "capture_holdout",
        "holdout_captures": HOLDOUT,
        "fast_mode": FAST,
        "seed": SEED,
        "git_sha": GIT_SHA,
    },
)

In [ ]:
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_A, y_label, test_size=HOLDOUT_FRACTION, random_state=SEED, stratify=y_label
)

print(f"same rows, split at random, {len(yp_test):,} test rows against {int(is_holdout.sum()):,} above")
print("fitting")

RESULTS["attack_random_pooled"] = runlog.fit_and_save(
    OUT_DIR,
    "attack_tier_a_random_pooled",
    RandomForestClassifier(n_estimators=N_TREES, random_state=SEED, n_jobs=-1),
    Xp_train,
    yp_train,
    Xp_test,
    yp_test,
    features=FEATURES,
    target="label",
    notes="tier A only, captures pooled and rows split at random",
    extra_config={"protocol": "random_pooled", "fast_mode": FAST, "seed": SEED, "git_sha": GIT_SHA},
)

In [ ]:
held = RESULTS["attack_capture_holdout"]["metrics"]
pooled = RESULTS["attack_random_pooled"]["metrics"]
MACRO_F1_GAP = pooled["macro_f1"] - held["macro_f1"]

print("classifying the attack, eight tier A classes, same model and same rows")
print()
print(f"{'':<22}{'capture held out':>18}{'rows pooled':>14}{'difference':>13}")
for metric in ("accuracy", "macro_f1", "weighted_f1"):
    print(f"{metric:<22}{held[metric]:>18.4f}{pooled[metric]:>14.4f}{pooled[metric] - held[metric]:>13.4f}")
print()
print(f"macro-F1 falls by {MACRO_F1_GAP:.4f} when the test recording is one the model never saw.")
print()

per_class = pd.DataFrame(
    {
        "capture_held_out": pd.Series(held["per_class_f1"]),
        "rows_pooled": pd.Series(pooled["per_class_f1"]),
    }
)
per_class["difference"] = per_class["rows_pooled"] - per_class["capture_held_out"]
per_class = per_class.sort_values("difference", ascending=False)

print("per-class F1")
print(per_class.to_string(float_format=lambda v: f"{v:,.4f}"))

## Step 8 — the verdict

Three questions, answered from the numbers above and nothing else: whether capture
identity is recoverable from the features, which family carries it, and what the
difference in macro-F1 was between a split that holds out a recording and one that does
not. The rule for the first was fixed in the configuration cell before any of this ran.

In [ ]:
capture = RESULTS["capture_id_all"]["metrics"]
over_chance = capture["accuracy"] / capture["chance_rate"]
RECOVERABLE = bool(
    capture["accuracy"] > RECOVERABLE_ACCURACY and over_chance > RECOVERABLE_MULTIPLE_OF_CHANCE
)

carrying = family_runs[CARRYING_FAMILY]["metrics"]
top_ranked = RANKING.head(5)["feature"].tolist()
top_within_class = RANKING.sort_values("within_class_median_ratio", ascending=False).head(5)["feature"].tolist()

statement = [
    (
        f"Capture identity is recoverable from the features. A forest given all {len(FEATURES)} "
        f"features names which of {capture['n_classes']} recordings a row came from with "
        f"{capture['accuracy']:.1%} accuracy and {capture['macro_f1']:.3f} macro-F1, against a "
        f"chance rate of {capture['chance_rate']:.1%}, which is {over_chance:,.0f} times chance."
        if RECOVERABLE
        else
        f"Capture identity is not recoverable at the threshold set in advance. A forest given all "
        f"{len(FEATURES)} features reaches {capture['accuracy']:.1%} accuracy on {capture['n_classes']} "
        f"recordings against a chance rate of {capture['chance_rate']:.1%}, which is "
        f"{over_chance:,.1f} times chance, below the rule of {RECOVERABLE_ACCURACY:.0%} and "
        f"{RECOVERABLE_MULTIPLE_OF_CHANCE:.0f} times chance."
    ),
    (
        f"With the attack class already known, the recording is still identifiable at "
        f"{WITHIN_CLASS_MEAN_ACC:.1%} accuracy averaged over the eight tier A classes, so this is "
        f"about the recording and not only about the attack."
    ),
    (
        f"Of the three families, {CARRYING_FAMILY} is the one that carries it: "
        f"{carrying['accuracy']:.1%} accuracy from its "
        f"{family_runs[CARRYING_FAMILY]['config']['n_features']} features alone, against "
        f"{capture['accuracy']:.1%} from all {len(FEATURES)}. The others reach "
        + ", ".join(
            f"{family} {family_runs[family]['metrics']['accuracy']:.1%}"
            for family in TEST_FAMILIES
            if family != CARRYING_FAMILY
        )
        + "."
    ),
    (
        f"The five features published work reports as most important reach "
        f"{named['accuracy']:.1%} on the same task, and IAT, Rate and Srate on their own reach "
        f"{timing['accuracy']:.1%}."
    ),
    (
        f"Classifying the attack on the eight tier A classes, macro-F1 is {held['macro_f1']:.4f} when "
        f"a whole recording is held out and {pooled['macro_f1']:.4f} when the same rows are pooled and "
        f"split at random. The difference of {MACRO_F1_GAP:.4f} is what the pooled protocol is worth "
        f"on the classes where a clean comparison is possible."
    ),
    (
        f"By between-capture variance the leading features are {', '.join(top_ranked)}. Holding the "
        f"class fixed, so that only recordings of the same attack are compared, they are "
        f"{', '.join(top_within_class)}."
    ),
]

if FAST:
    statement.append(
        "FAST=1. Each file was read from its opening rows only, which is a contiguous slice of a "
        "session rather than a sample of it, so every number here is a screen and none of it "
        "belongs in the ledger."
    )

verdict = {
    "generated_by": "AG_PRAXIS_NB04_leakage_diagnostic.ipynb",
    "generated_on": RUN_DATE,
    "git_sha": GIT_SHA,
    "fast_mode": FAST,
    "read_fraction": READ_FRAC,
    "seed": SEED,
    "rows_per_capture_for_models": ROWS_PER_CAPTURE,
    "rule": {
        "recoverable_if_accuracy_above": RECOVERABLE_ACCURACY,
        "and_multiple_of_chance_above": RECOVERABLE_MULTIPLE_OF_CHANCE,
        "fixed_before_running": True,
    },
    "capture_identity_recoverable": RECOVERABLE,
    "capture_identification": {
        "n_captures": capture["n_classes"],
        "accuracy": capture["accuracy"],
        "macro_f1": capture["macro_f1"],
        "chance_rate": capture["chance_rate"],
        "multiple_of_chance": over_chance,
        "top_features": [e["feature"] for e in capture["top_features"]],
    },
    "capture_identification_within_class": {
        "mean_accuracy": WITHIN_CLASS_MEAN_ACC,
        "per_class": {
            label: {
                "recordings": run["metrics"]["n_classes"],
                "accuracy": run["metrics"]["accuracy"],
                "chance_rate": run["metrics"]["chance_rate"],
            }
            for label, run in within_class_runs.items()
        },
    },
    "carrying_family": CARRYING_FAMILY,
    "family_accuracy": {
        family: {
            "n_features": run["config"]["n_features"],
            "accuracy": run["metrics"]["accuracy"],
            "macro_f1": run["metrics"]["macro_f1"],
        }
        for family, run in family_runs.items()
    },
    "named_features": {
        "five": {"features": NAMED_SUSPECTS, "accuracy": named["accuracy"], "macro_f1": named["macro_f1"]},
        "timing_three": {"features": TIMING_THREE, "accuracy": timing["accuracy"], "macro_f1": timing["macro_f1"]},
    },
    "attack_classification_tier_a": {
        "classes": held["n_classes"],
        "capture_holdout": {
            "macro_f1": held["macro_f1"],
            "weighted_f1": held["weighted_f1"],
            "accuracy": held["accuracy"],
            "holdout_captures": HOLDOUT,
        },
        "random_pooled": {
            "macro_f1": pooled["macro_f1"],
            "weighted_f1": pooled["weighted_f1"],
            "accuracy": pooled["accuracy"],
            "test_fraction": HOLDOUT_FRACTION,
        },
        "macro_f1_difference": MACRO_F1_GAP,
    },
    "variance_ranking_top_5": top_ranked,
    "variance_ranking_top_5_within_class": top_within_class,
    "statement": statement,
}

VERDICT_PATH = OUT_DIR / "NB04_verdict.json"
VERDICT_PATH.write_text(json.dumps(cap.jsonable(verdict), indent=2, default=str) + "\n")

print("=" * 79)
print("VERDICT")
print("=" * 79)
for line in statement:
    print()
    print(line)
print()
print("=" * 79)
print(f"wrote {VERDICT_PATH}   {VERDICT_PATH.stat().st_size:,} bytes")

Every run this notebook fitted, listed with where it went. Each directory holds the same
five files: the configuration, the metrics, the true labels, the predicted labels, and
the model. The models here are scikit-learn forests rather than Keras, so the fifth file
is `model.joblib`.

In [ ]:
run_dirs = sorted(p for p in OUT_DIR.iterdir() if p.is_dir())
rows = []
for path in run_dirs:
    metrics = json.loads((path / "metrics.json").read_text())
    config = json.loads((path / "config.json").read_text())
    rows.append(
        {
            "run": path.name,
            "target": config["target"],
            "features": config["n_features"],
            "test_rows": metrics["n_test"],
            "classes": metrics["n_classes"],
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "files": len(list(path.iterdir())),
        }
    )

print(f"{len(run_dirs)} runs written to {OUT_DIR}")
print()
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

Files on Drive are one copy. The output of a cell saved with the notebook is another, and
it is the one that stays attached to the run that produced it. So the two files that
carry the result are read back from disk and printed in full.

In [ ]:
for path in (VARIANCE_PATH, VERDICT_PATH):
    print("=" * 79)
    print(f"{path.name}   ({path.stat().st_size:,} bytes)")
    print("=" * 79)
    print(path.read_text().rstrip())
    print()

The entry for `RESULTS_LEDGER.md`, ready to paste.

In [ ]:
status = "reference run" if not FAST else "EXPLORATORY, FAST=1, screen only, do not enter in the ledger"

ledger = f'''
### NB04 — leakage diagnostic ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB04_leakage_diagnostic.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| FAST | {int(FAST)} |
| status | {status} |
| rows read | {rows_read:,} of {sum(ROW_COUNTS.values()):,} |
| rows per capture kept for models | {ROWS_PER_CAPTURE:,} |
| features | {len(FEATURES)}, after dropping {", ".join(DROPPED)} |
| model | RandomForest, {N_TREES} trees, seed {SEED} |
| top feature by between-capture variance | {RANKING.iloc[0]["feature"]}, ratio {RANKING.iloc[0]["ratio"]:,.2f} |
| top feature within class | {top_within_class[0]} |
| capture identification, all features | accuracy {capture["accuracy"]:.4f}, macro-F1 {capture["macro_f1"]:.4f}, {capture["n_classes"]} captures, chance {capture["chance_rate"]:.4f} |
| capture identification, class held fixed | mean accuracy {WITHIN_CLASS_MEAN_ACC:.4f} over 8 classes |
| carrying family | {CARRYING_FAMILY}, accuracy {carrying["accuracy"]:.4f} from {family_runs[CARRYING_FAMILY]["config"]["n_features"]} features |
| the five named features | accuracy {named["accuracy"]:.4f} |
| IAT, Rate, Srate | accuracy {timing["accuracy"]:.4f} |
| attack macro-F1, capture held out | {held["macro_f1"]:.4f} |
| attack macro-F1, rows pooled | {pooled["macro_f1"]:.4f} |
| macro-F1 difference | {MACRO_F1_GAP:.4f} |
| capture identity recoverable | {RECOVERABLE} |
| artefacts | {OUT_DIR}, holding variance_ranking.json, NB04_verdict.json and {len(run_dirs)} run directories |

Held out for the capture-disjoint fit: {", ".join(f"{k} -> {v}" for k, v in sorted(HOLDOUT.items()))}
'''

print(ledger)